# Module 5: Deploy to AgentCore Runtime (15 min)

Deploy the customer service agent you built in Modules 1-4 to **Amazon Bedrock AgentCore Runtime** - a managed runtime for hosting agents with no servers to manage. You bring your existing `main.py`; the `agentcore` CLI packages and deploys it.

**Prerequisites:** Modules 1-4 completed, AWS credentials with AgentCore access, the `agentcore` CLI installed (`sudo npm install -g @aws/agentcore`; already installed in Workshop Studio).

> The deploy steps run in a **terminal**, not in notebook cells - `agentcore` is an interactive CLI (it prompts you and streams progress). Open a terminal in Code Editor (**Terminal -> New Terminal**) and run the commands below from `/workshop/samples/05-deploy`.

---

## Part 1: The Deployment Code (`main.py`)

The only change from a local agent is the **entry point**. Locally you call `agent(prompt)` in a loop; on AgentCore you expose a handler that receives a `payload` and returns a response. The workshop's `main.py` wraps the same agent - same tools, same steering handlers, same conversation manager - behind that handler.

```python
from bedrock_agentcore.runtime import BedrockAgentCoreApp

app = BedrockAgentCoreApp()

@app.entrypoint
def invoke(payload, context):
    prompt = payload.get("prompt")
    if not prompt:
        raise ValueError("Missing required field: prompt")
    agent = get_agent()
    response = agent(prompt)
    return str(response).strip()

if __name__ == "__main__":
    app.run()
```

- `BedrockAgentCoreApp()` creates the runtime app.
- `@app.entrypoint` marks the handler AgentCore calls on each invocation - it receives `payload` (a dict with the user prompt) and `context`.
- Returning `str(response).strip()` sends back the agent's reply as plain text.
- `app.run()` lets you run the same file locally as an HTTP server.

Open `main.py` in this folder to see the full agent (it imports `customer_service_tools.py` and `steering_handlers.py` - the same code from earlier modules).

---

## Part 2: Test the Agent Locally (optional)

Before deploying, you can exercise the agent logic directly in this notebook. The cell below builds the same agent `main.py` deploys and runs one prompt.

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
from main import get_agent

agent = get_agent()
result = agent("Hi, I'm customer C-1001. What are my recent orders?")
print(f"\nTokens: {result.metrics.accumulated_usage['totalTokens']}")

---

## Part 3: Deploy with the AgentCore CLI

The rest of this module runs in a **terminal** from `/workshop/samples/05-deploy`. The `agentcore` CLI comes from the `@aws/agentcore` npm package.

### Step 1 - Move into the module folder

```bash
cd /workshop/samples/05-deploy
```

Your agent code is already here: `main.py`, `customer_service_tools.py`, `steering_handlers.py`, `skills/`, and `requirements.txt`.

### Step 2 - Create the AgentCore project

```bash
agentcore create
```

`agentcore create` is **interactive**:

1. **Project name** - accept the default or type a name (e.g. `csdeploy`).
2. **What would you like to build?** - choose **Skip** ("I'll add resources later").

This generates the project shell with an `agentcore/` folder (config + CDK infrastructure). It does **not** generate agent code yet - you bring your own in the next step.

### Step 3 - Add your agent (Bring Your Own Code)

Move into the project folder that `create` made, then add the agent:

```bash
cd csdeploy      # the project name you chose
agentcore add
```

`agentcore add` walks through a short wizard - choose **agent**, then:

| Prompt | Choose |
|--------|--------|
| **Name** | `MyAgent` |
| **Type** | **Bring my own code** |
| **Code location** | press **Enter** for the default `app/MyAgent/` |
| **Entrypoint** | press **Enter** for the default `main.py` |
| **Build** | **Direct Code Deploy** (zips your code to S3 - no container) |
| **Model provider** | **Amazon Bedrock** |
| **Advanced** | press **Enter** to accept defaults |
| **Confirm** | review and press **Enter** |

On success the CLI reminds you: **"Copy your agent code to `app/MyAgent/` before deploying."**

### Step 4 - Copy your agent code into `app/MyAgent/`

`add` created an empty `app/MyAgent/` folder. Copy the agent you built into it - `main.py` plus the files it imports:

```bash
cp ../main.py ../customer_service_tools.py ../steering_handlers.py ../requirements.txt app/MyAgent/
cp -r ../skills app/MyAgent/
```

The `entrypoint` (`main.py`) must sit at the root of that folder.

### Step 4b - Set up dependencies in `app/MyAgent/`

`agentcore deploy` builds the agent from a **`pyproject.toml`** in `app/MyAgent/` (a plain `requirements.txt` is not enough - the CDK build needs the `pyproject.toml`). Generate it with `uv` and add the dependencies:

```bash
cd app/MyAgent
uv init --bare --python 3.13
uv add strands-agents bedrock-agentcore aws-opentelemetry-distro boto3
cd ../..
```

`uv init --bare` creates the `pyproject.toml` without overwriting your `main.py`; `uv add` writes the dependencies.

> If you skip this, `agentcore deploy` fails with `Required project file not found: .../pyproject.toml`.

### Step 5 - Test locally with `agentcore dev` (local machine only)

> **This step only works when you run the workshop on your own computer, not in AWS Workshop Studio.** `agentcore dev` starts a local web UI and tries to open a browser; the Workshop Studio Code Editor environment has no browser/display (and no `xdg-open`), so it can't show the inspector there. On Workshop Studio, skip straight to Step 6 (Deploy) and test with `agentcore invoke`.

On a local machine:

```bash
agentcore dev
```

This starts a local server and opens the **agent inspector** in your browser (`http://localhost:8081`). Chat with the agent there to confirm it works, then press **Ctrl+C** to stop.

### Step 6 - Deploy to AgentCore Runtime

```bash
agentcore deploy
```

`agentcore deploy` validates the project, synthesizes CloudFormation, zips your code to an Amazon S3 staging bucket, and provisions the AgentCore Runtime. The first deploy takes a few minutes (it installs dependencies); later updates reuse the cached dependencies and are faster.

### Step 7 - Invoke the deployed agent

```bash
agentcore invoke "Hi, I'm customer C-1001. What are my recent orders?"
```

The prompt is passed as a positional argument. Use `--session-id` to keep context across calls. The session ID must be **at least 33 characters** long (a shorter one fails with `Value at 'runtimeSessionId' failed to satisfy constraint: Member must have length greater than or equal to 33`):

```bash
agentcore invoke --session-id customer-1001-session-refund-000001 "I need a refund for order ORD-5521"
```

Run `agentcore invoke` with no prompt to open an interactive chat against the deployed agent.

### Step 8 - Invoke from code with boto3 (the production path)

`agentcore invoke` is for testing from the terminal. In production you call the deployed runtime directly with the **AWS SDK** (`InvokeAgentRuntime`). Same agent, two ways to reach it.

Get the runtime ARN with `agentcore status` - it prints something like `arn:aws:bedrock-agentcore:us-east-1:<account>:runtime/<project>_<name>-XXXXXXXXXX`. Paste it into the cell below and run it.

In [ ]:
import json
import uuid
import boto3

# Paste the ARN from `agentcore status`:
agent_arn = "arn:aws:bedrock-agentcore:us-east-1:<account>:runtime/<project>_<name>-XXXXXXXXXX"
prompt = "Hi, I'm customer C-1001. What are my recent orders?"

client = boto3.client("bedrock-agentcore", region_name="us-east-1")

response = client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    runtimeSessionId=str(uuid.uuid4()),  # must be 33+ chars; a uuid4 satisfies that
    payload=json.dumps({"prompt": prompt}).encode(),
    qualifier="DEFAULT",
)

# The response body streams back in chunks - join and decode them
chunks = [chunk.decode("utf-8") for chunk in response.get("response", [])]
print("".join(chunks))

---

## Troubleshooting

| Symptom | Resolution |
|---------|------------|
| `unknown command 'configure'` | You have the **old** CLI. `configure` was replaced - use `agentcore create` + `agentcore add`. Uninstall the old toolkit: `pip uninstall bedrock-agentcore-starter-toolkit`. |
| `agentcore deploy` fails with `Required project file not found: .../pyproject.toml` | The agent folder (`app/MyAgent/`) has no `pyproject.toml`. Run Step 4b: `cd app/MyAgent && uv init --bare --python 3.13 && uv add strands-agents bedrock-agentcore aws-opentelemetry-distro boto3`. |
| `npm error code EACCES` on `npm install -g` | A global npm install needs root. Use `sudo npm install -g @aws/agentcore`. (In Workshop Studio it is already installed.) |
| `agentcore deploy` fails on region | The region comes from `agentcore/aws-targets.json`. Make sure its `region` is `us-east-1`. |
| `AccessDeniedException` on Bedrock | The first call to a model auto-initiates access in the background; wait a couple of minutes and retry. |
| `Value at 'runtimeSessionId' failed to satisfy constraint: Member must have length greater than or equal to 33` | The `--session-id` you passed to `agentcore invoke` is too short - AgentCore requires session IDs of **33+ characters**. Use a longer ID (e.g. `customer-1001-session-refund-000001`) or a UUID. |

> The Workshop Studio environment is already CDK-bootstrapped, so `agentcore deploy` works without any `cdk bootstrap` step. (If you run this locally instead, `agentcore deploy` bootstraps your account automatically on first use.)

---

## Cleanup

When you're done, tear down the AWS resources this module created. Cleanup is **two steps**: reset the project config, then deploy that empty state so AWS removes the resources.

```bash
agentcore remove all -y    # clears the local config (does NOT touch AWS yet)
agentcore deploy           # applies the teardown - removes the runtime from AWS
```

`agentcore remove all -y` only resets `agentcore/agentcore.json` locally; the follow-up `agentcore deploy` is what actually deletes the AgentCore Runtime and its CloudFormation stack.

---

## Core Path Complete!

You've built and deployed a production-ready customer service agent from scratch:

1. ✅ **Agent Loop + Tools** - core agent with customer service capabilities
2. ✅ **Hooks** - rate limiting via deterministic code in the loop
3. ✅ **Skills + Steering** - workflow knowledge and business-rule enforcement
4. ✅ **Session Managers** - persistent memory across restarts
5. ✅ **Deploy** - production deployment on AgentCore Runtime

### Optional next modules

Two optional modules extend this same agent:

- **Module 6: Multi-Agent** - delegate technical issues to a specialist agent
- **Module 7: Evals** - measure agent quality automatically with LLM-as-a-judge

### Resources

- [AgentCore CLI](https://github.com/aws/agentcore-cli)
- [Direct code deployment for Python](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-get-started-code-deploy-python.html)
- [Strands Agents Documentation](https://strandsagents.com)